# 02 — Preprocessing


Goals:
- Define predictors and target
- Decide which features to retain/exclude
- Separate numeric and categorical columns
- Create train/test split
- Build preprocessing pipeline


02 — Bank Marketing Data Preprocessing
Dataset: bank-full.csv (Portuguese bank marketing campaigns)
Task: Predict whether a client subscribes to a term deposit (y).
Preprocessing goals
1. Define predictors and target.
2. Decide which features to retain or exclude.
3. Separate numerical and categorical columns.
4. Create a train/test split.
5. Build a reusable Scikit-learn preprocessing pipeline.
Modeling assumption: We aim to make predictions before the phone call, so duration is excluded. The other feature exclusions are initial modeling choices to evaluate later.

1. Import Libraries
Import the packages needed for data manipulation, train/test splitting, preprocessing, validation, and saving the fitted transformer.

In [36]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


2. Load and Explore the Dataset
Load the semicolon-delimited bank-full.csv file and inspect its shape and first few rows.

In [37]:
# TODO: load data and define X / y
# Load the full Bank Marketing dataset
df = pd.read_csv("../data/bank-full.csv", sep=";")
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


We have 45211 observations, 16 input features, 1 target variable.

2.1 Examine Target Distribution
The target y records subscription (yes) or non-subscription (no). Encode it as 1 and 0, then inspect the class counts and percentages. The dataset is imbalanced, so accuracy alone will not be sufficient for later model evaluation.

In [38]:
# Convert target into binary values
y = df["y"].map({"yes": 1, "no": 0})

print("Target distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print(y.value_counts(normalize=True) * 100)

Target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Target distribution (%):
y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


3. Define Predictors and Target

The target is binary y. We exclude five features from the initial model:

Excluded feature	Rationale
duration	        Call length is not known before the call; including it would introduce information unavailable at prediction time.
default	            Initially omitted due to potentially limited predictive contribution; this is not yet empirically established.
day	                Initially omitted to simplify the model; its predictive contribution should be tested.
pdays	            Omitted to avoid handling the -1 sentinel and to test a simpler model without this previous-campaign measure.
poutcome	        Omitted to test a model without previous-campaign outcome information.


5 features (default, day, duration, pdays, and poutcome) were excluded during feature selection. 
The duration attribute was removed to prevent information leakage because call duration is unavailable before contacting a customer. Including it would introduce information unavailable at prediction time.
The default and day attributes were excluded based on their potentially limited predictive contribution.
The pdays and poutcome were excluded to simplify the initial model and evaluate performance without previous-campaign information. 

After feature selection, 11 predictor variables remained for model development. 
The impact of these exclusions will be assessed during model evaluation.

In [39]:
# Define target variable (1 = subscribed, 0 = did not subscribe)
y = df["y"].map({"yes": 1, "no": 0})

# Features to exclude
features_to_exclude = [
    "default",
    "day",
    "duration",
    "pdays",
    "poutcome"
]

# Define predictors
X = df.drop(columns=["y"] + features_to_exclude)

# Display results
print("Predictors shape:", X.shape)
print("Target shape:", y.shape)

print("\nSelected predictors:")
print(X.columns.tolist())

Predictors shape: (45211, 11)
Target shape: (45211,)

Selected predictors:
['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'contact', 'month', 'campaign', 'previous']


3.1 Verify Feature Selection
Confirm that X contains 11 predictors, the excluded columns are absent, and the target has no unmapped values.

In [40]:
print("X shape:", X.shape)
print("y shape:", y.shape)

assert "y" not in X.columns
assert all(col not in X.columns for col in ["y"] + features_to_exclude)
assert y.notna().all()
assert set(y.unique()) == {0, 1}
assert len(X) == len(y)

print("Predictors and target defined successfully!")

X shape: (45211, 11)
y shape: (45211,)
Predictors and target defined successfully!


4. Separate Numerical and Categorical Features
Identify feature types automatically. The expected groups are 4 numerical (age, balance, campaign, previous) and 7 categorical (job, marital, education, housing, loan, contact, month). Numerical features will be scaled, while categorical features will be one-hot encoded.

After feature selection, the remaining 11 predictors were separated into numerical and categorical variables based on their data types. Four numerical features (age, balance, campaign, and previous) were identified for scaling, while seven categorical features (job, marital, education, housing, loan, contact, and month) were identified for one-hot encoding. This separation allows appropriate preprocessing techniques to be applied to each feature type through a reusable Scikit-learn pipeline.

In [41]:
# Identify numerical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical features
categorical_features = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical Features: ['age', 'balance', 'campaign', 'previous']
Categorical Features: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month']

Number of numerical features: 4
Number of categorical features: 7


In [42]:
# Verify all 11 features are correctly classified
assert len(numerical_features) == 4
assert len(categorical_features) == 7

assert len(numerical_features) + len(categorical_features) == X.shape[1]

assert set(numerical_features).isdisjoint(categorical_features)

print("All 11 features classified successfully!")

All 11 features classified successfully!


5. Inspect Missing, Unknown, and Invalid Values
Check for actual missing values (NaN) and the literal category "unknown". Keep "unknown" as its own category for this initial model. Imputation of actual missing values will occur inside the pipeline and will be learned from training data only.

5.1 Missing Values
Count missing values and report their percentage for each selected predictor.

In [43]:
#1. Check for missing values (NaN)
# Check missing values in selected predictors
missing_values = X.isnull().sum()

missing_percentage = (X.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage (%)": missing_percentage
})

display(missing_summary)

,Missing Count,Missing Percentage (%)
age,0,0.0
job,0,0.0
marital,0,0.0
education,0,0.0
balance,0,0.0
housing,0,0.0
loan,0,0.0
contact,0,0.0
month,0,0.0
campaign,0,0.0


5.2 Unknown Categorical Values
Count the literal "unknown" values separately from actual missing values. One-hot encoding will preserve this category when it occurs in training data.

Before creating our preprocessing pipeline, we should check for actual missing values (NaN) and categorical values such as "unknown". We will decide how to handle them without unnecessarily removing observations. We don't have any missing values. But we do have "unknown" in the categorical features so for now we will keep "unknown" as a separate cateogry and oneHotEncoder can handle unknown as a regular category. We can later compare model performance with alternative missing-value strategies.

In [44]:
#2. Identify unknown values in categorical features
# Count unknown values in categorical features
unknown_counts = (
    X[categorical_features]
    .eq("unknown")
    .sum()
)

unknown_percentage = (
    X[categorical_features]
    .eq("unknown")
    .mean() * 100
).round(2)

unknown_summary = pd.DataFrame({
    "Unknown Count": unknown_counts,
    "Unknown Percentage (%)": unknown_percentage
})

display(unknown_summary)

,Unknown Count,Unknown Percentage (%)
job,288,0.64
marital,0,0.00
education,1857,4.11
housing,0,0.00
loan,0,0.00
contact,13020,28.80
month,0,0.00


5.3 Numerical Validity Checks
Inspect numerical summaries and infinite values. Negative balance values and previous = 0 can be valid and should not automatically be removed. 

Check whether the numerical features contain missing or infinite values. Negative values in balance are valid and should not automatically be treated as missing. They can indicate a negative average account balance. Also, previous = 0 is a valid value indicating that the customer had no previous campaign contacts.

In [45]:
#Verify numerical features
import numpy as np

# Check numerical features
numerical_summary = X[numerical_features].describe()

display(numerical_summary)

# Check infinite values
infinite_counts = pd.Series(
    np.isinf(X[numerical_features].to_numpy()).sum(axis=0),
    index=numerical_features
)

print("\nInfinite values:")
print(infinite_counts)

,age,balance,campaign,previous
count,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,2.763841,0.580323
std,10.618762,3044.765829,3.098021,2.303441
min,18.000000,-8019.000000,1.000000,0.000000
25%,33.000000,72.000000,1.000000,0.000000
50%,39.000000,448.000000,2.000000,0.000000
75%,48.000000,1428.000000,3.000000,0.000000
max,95.000000,102127.000000,63.000000,275.000000



Infinite values:
age         0
balance     0
campaign    0
previous    0
dtype: int64


5.4 Duplicate Row Check
Inspect exact duplicates in the source dataset. Do not automatically delete them: identical records do not necessarily identify the same customer.

In [46]:
# Check duplicate observations
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 0


5.5 Missing-Value Summary and Decision
Summarize the checks. Preserve the original rows; use median imputation for missing numerical values and most-frequent imputation for missing categorical values inside the fitted preprocessing pipeline.

In [47]:
print("Total observations:", len(X))
print("Total selected features:", X.shape[1])

print("\nTotal missing values:", X.isnull().sum().sum())

print(
    "Total unknown categorical values:",
    X[categorical_features].eq("unknown").sum().sum()
)

print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Total observations: 45211
Total selected features: 11

Total missing values: 0
Total unknown categorical values: 15165

Numerical features: 4
Categorical features: 7


Our preprocessing decision
For now, we will not manually replace or remove any observations.
Instead, our reusable Scikit-learn preprocessing pipeline will:
- Use SimpleImputer(strategy="median") for actual missing numerical values.
- Use SimpleImputer(strategy="most_frequent") for actual missing categorical values.
- Preserve the literal "unknown" category.
- Apply StandardScaler to numerical variables.
- Apply OneHotEncoder(handle_unknown="ignore") to categorical variables.
These transformations will be fitted only on training data to prevent data leakage.

The selected predictor variables were examined for missing values and unknown categorical entries. The dataset does not contain explicitly recorded missing values, but certain categorical features include the value "unknown". These values were retained as separate categories to preserve potentially useful information and avoid unnecessary data loss. Numerical features were also examined for invalid values. Missing-value imputation will be incorporated into the preprocessing pipeline to ensure the model can handle incomplete observations consistently.


6. Create the Train/Test Split
Use an 80/20 stratified split with random_state=42. Stratification preserves the approximate subscriber/non-subscriber proportions in both sets. The test set remains untouched while preprocessing parameters are learned from training data.

In [48]:
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display dataset shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (36168, 11)
X_test shape: (9043, 11)
y_train shape: (36168,)
y_test shape: (9043,)


6.1 Check Class Distribution
Compare class proportions in the original, training, and testing datasets to confirm stratification.

In [49]:
# Compare class distributions
class_distribution = pd.DataFrame({
    "Original Dataset": y.value_counts(normalize=True) * 100,
    "Training Set": y_train.value_counts(normalize=True) * 100,
    "Testing Set": y_test.value_counts(normalize=True) * 100
}).round(2)

class_distribution.index = ["No Subscription (0)", "Subscription (1)"]

display(class_distribution)

,Original Dataset,Training Set,Testing Set
No Subscription (0),88.3,88.3,88.3
Subscription (1),11.7,11.7,11.7


6.2 Validate the Split
Verify sizes, target alignment, disjoint original row indices, and similar positive-class proportions.


The Bank Marketing dataset was divided into training and testing subsets using an 80/20 split. Stratified sampling was applied to preserve the original class distribution because the dataset is imbalanced, with approximately 88.3% non-subscribers and 11.7% subscribers. The training set contains 36,168 observations, while the testing set contains 9,043 observations. A fixed random state of 42 was used to ensure reproducibility. All preprocessing transformations will be fitted exclusively on the training set to prevent data leakage.

In [51]:
# Verify training and testing sizes
assert len(X_train) + len(X_test) == len(X)

# Verify number of selected features
assert X_train.shape[1] == 11
assert X_test.shape[1] == 11

# Verify target and predictors are aligned
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Verify no overlapping observations by original row index
assert set(X_train.index).isdisjoint(set(X_test.index))

# Verify similar class proportions
assert abs(y_train.mean() - y_test.mean()) < 0.01

print("Train/test split completed successfully!")

Train/test split completed successfully!


7. Build the Reusable Preprocessing Pipeline
Create two preprocessing branches:
- Numerical: median imputation → StandardScaler.
- Categorical: most-frequent imputation → OneHotEncoder(handle_unknown="ignore").
Combine them with ColumnTransformer. Fit transformations only on X_train and apply the fitted transformer to X_test. The literal "unknown" is retained as a category; handle_unknown="ignore" refers to previously unseen categories at inference time.

In [52]:
# ------------------------------------------------------------
# 1. Define Numerical and Categorical Preprocessing Pipelines
# ------------------------------------------------------------

# Numerical: Handle missing values and standardize
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: Handle missing values and one-hot encode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# ------------------------------------------------------------
# 2. Combine Pipelines Using ColumnTransformer
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# ------------------------------------------------------------
# 3. Fit on Training Data and Transform Testing Data
# ------------------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# ------------------------------------------------------------
# 4. Convert Processed Arrays into DataFrames
# ------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)


# ------------------------------------------------------------
# 5. Display Preprocessing Results
# ------------------------------------------------------------

print("Original Training Shape:", X_train.shape)
print("Processed Training Shape:", X_train_processed_df.shape)

print("\nOriginal Testing Shape:", X_test.shape)
print("Processed Testing Shape:", X_test_processed_df.shape)

print("\nTotal Features After Encoding:", len(feature_names))

print("\nProcessed Training Data (First 5 Rows):")
display(X_train_processed_df.head())


# ------------------------------------------------------------
# 6. Validate Preprocessing
# ------------------------------------------------------------

assert X_train_processed_df.shape[0] == X_train.shape[0]
assert X_test_processed_df.shape[0] == X_test.shape[0]

assert X_train_processed_df.shape[1] == X_test_processed_df.shape[1]

assert X_train_processed_df.columns.equals(
    X_test_processed_df.columns
)

assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

assert X_train.shape[1] == 11
assert X_test.shape[1] == 11

print("\nPreprocessing validation completed successfully!")


# ------------------------------------------------------------
# 7. Save Fitted Preprocessing Pipeline
# ------------------------------------------------------------

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    models_dir / "bank_preprocessor.joblib"
)

print("\nPreprocessing pipeline saved successfully!")

Original Training Shape: (36168, 11)
Processed Training Shape: (36168, 42)

Original Testing Shape: (9043, 11)
Processed Testing Shape: (9043, 42)

Total Features After Encoding: 42

Processed Training Data (First 5 Rows):


,num__age,num__balance,num__campaign,num__previous,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,...,cat__month_dec,cat__month_feb,cat__month_jan,cat__month_jul,cat__month_jun,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep
24001,-0.460434,-0.164410,-0.246104,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43409,-1.589641,0.899627,0.398202,2.664584,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20669,0.292371,-0.365486,0.398202,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18810,0.668773,-0.445003,2.653271,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
23130,-0.272233,-0.361249,2.331118,-0.241509,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Preprocessing validation completed successfully!

Preprocessing pipeline saved successfully!


A reusable preprocessing pipeline was developed using Scikit-learn's Pipeline and ColumnTransformer classes. Numerical features were processed using median imputation and StandardScaler, while categorical features were processed using most-frequent imputation and OneHotEncoder. Unknown categorical values were retained as separate categories, and previously unseen categories were handled using handle_unknown="ignore". The preprocessing pipeline was fitted exclusively on the training dataset and subsequently applied to the testing dataset to prevent data leakage. The fitted pipeline was saved using Joblib for reuse during model development and evaluation.



8. Preprocessing Summary
The Bank Marketing dataset was prepared for binary classification. The target y was encoded as 0/1; five initial predictors were excluded, leaving 11 predictors (4 numerical and 7 categorical). Actual missing values were checked, and literal "unknown" categories were retained. An 80/20 stratified train/test split preserved class proportions. A reusable ColumnTransformer applied median imputation and scaling to numerical features, and most-frequent imputation and one-hot encoding to categorical features. The transformer was fitted only on the training set, validated on both partitions, and saved for reuse.
Next stage: Compare classification models using training-only cross-validation, then evaluate the selected model on the held-out test set. Consider revisiting the four non-leakage-related feature exclusions as an experiment.

In [ ]:
# TODO: document the final feature inclusion/exclusion decisions
